Following my process in R for the HS data, now just here in Python for the transfer and JC data.

## Imports

In [1]:
import pandas as pd

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

## Load and Prep Data

In [ ]:
data = pd.read_excel('RecruitmentPrediction.xlsx', sheet_name='Transfers')

data = data[[
    'Years', '247Top', 'Position', 'Distance', 'Conf', 'Height', 'Weight',
    'Score', 'LDS', 'Alumni', 'Prev', 'Poly', 'BYU'
]]

categorical_cols = ['247Top', 'Position', 'Conf', 'LDS', 'Alumni', 'Prev', 'Poly']

for col in categorical_cols:
    data[col] = data[col].astype('category')

data['BYU'] = data['BYU'].map({'N': 0, 'Y': 1})

X = data.drop(columns='BYU')
y = data['BYU']

data.head()

,Years,247Top,Position,Distance,Conf,Height,Weight,Score,LDS,Alumni,Prev,Poly,BYU
0,1,Y,RB,343.99,G6I,74,220,0.92,N,N,Y,N,1
1,1,Y,QB,1780.11,ACC,74,215,0.91,N,N,N,N,0
2,1,Y,CB,70.46,FCS,70,190,0.91,N,N,N,N,1
3,2,Y,DL,1856.74,B1G,74,298,0.90,N,N,N,Y,0
4,3,Y,LB,642.56,B1G,74,200,0.90,Y,N,N,N,1


## Preprocessing

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

## Pipeline

In [7]:
pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=123)),
    ('model', RandomForestClassifier(random_state=123))
])

## Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

param_grid = {
    'model__n_estimators': [300, 425, 675],
    'model__max_features': [5, 8, 10],
    'model__min_samples_leaf': [20, 40, 60]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(X, y)

print('Best parameters:', grid_search.best_params_)
print('Best ROC AUC:', grid_search.best_score_)

Best parameters: {'model__max_features': 8, 'model__min_samples_leaf': 20, 'model__n_estimators': 425}
Best ROC AUC: 0.8086969696969698


In [10]:
import joblib
joblib.dump(grid_search.best_estimator_, "transfer_jc_model.pkl")

['transfer_jc_model.pkl']